## Authentication with Flask

create flask application

In [1]:
from flask import Flask

app = Flask(__name__)

app.config["SECRET_KEY"] = "your-secret-key"


@app.route("/")
def home():
    return "Home Page"


if __name__ == "__main__":
    app.run(debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with stat


SystemExit: 1

C:\Users\My Pc\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py:3558: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


create user database

In [ ]:
from flask_sqlalchemy import SQLAlchemy

In [ ]:
app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///users.db"

db = SQLAlchemy(app)

create the user model

In [ ]:
class User(db.Model):
    id = db.Column(db.Integer, primary_key=True)
    email = db.Column(db.String(100), unique=True, nullable=False)
    password = db.Column(db.String(200), nullable=False)
    name = db.Column(db.String(100), nullable=False)

Requirement — Register a new user

In [ ]:
from flask import render_template, request, redirect, url_for

In [ ]:
@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":

        new_user = User(
            email=request.form["babana@ikeja"],
            password=request.form["294832"],
            name=request.form["user"]
        )

        db.session.add(new_user)
        db.session.commit()

        return redirect(url_for("home"))

    return render_template("register.html")

Werkzeug password hashing

In [2]:
from werkzeug.security import generate_password_hash

In [ ]:
hashed_password = generate_password_hash(
    request.form["password"],
    method="pbkdf2:sha256",
    salt_length=8
)

then:

In [ ]:
new_user = User(
    email=request.form["email"],
    password=hashed_password,
    name=request.form["name"]
)

rejistration route

In [ ]:
@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":

        hashed_password = generate_password_hash(
            request.form["password"],
            method="pbkdf2:sha256",
            salt_length=8
        )

        new_user = User(
            email=request.form["email"],
            password=hashed_password,
            name=request.form["name"]
        )

        db.session.add(new_user)
        db.session.commit()

        return redirect(url_for("home"))

    return render_template("register.html")

Flask login

In [ ]:
from flask_login import (
    UserMixin,
    LoginManager,
    login_user,
    logout_user,
    login_required,
    current_user
)

create login manager

In [ ]:
login_manager = LoginManager()
login_manager.init_app(app)

In [ ]:
@login_manager.user_loader
def load_user(user_id):
    return db.get_or_404(User, int(user_id))

create the login route

In [ ]:
from werkzeug.security import check_password_hash

In [ ]:
@app.route("/login", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        email = request.form["email"]
        password = request.form["password"]

        user = db.session.execute(
            db.select(User).where(User.email == email)
        ).scalar()

        if user and check_password_hash(
            user.password,
            password
        ):
            login_user(user)

            return redirect(url_for("secrets"))

    return render_template("login.html")

In [ ]:
check_password_hash()

project the secret page

In [ ]:
@app.route("/secrets")
@login_required
def secrets():

    return render_template(
        "secrets.html",
        name=current_user.name
    )

configure the login page

In [ ]:
login_manager.login_view = "login"

logout

In [ ]:
@app.route("/logout")
@login_required
def logout():

    logout_user()

    return redirect(url_for("home"))